# Phase 1: one topic in, one brief out

Runs the phase 1 graph, `planner -> researcher -> writer`, on one real topic, and keeps the record of the run:

1. the brief, rendered below and saved to `notebooks/phase_01_brief.md`;
2. the run numbers (sub-questions, findings, sections, retries, time);
3. the link to the run's trace in LangSmith.

Needs real keys in `.env`. One run costs up to 8 model calls and 6 Tavily searches, and takes about a minute.

Outputs are stripped from this file when it is committed (`nbstripout`); the brief file is the committed record. Concept: `tracing-from-day-one`.

In [ ]:
# Work from the project root, wherever Jupyter started: settings.py reads .env
# from the working folder, and the graph package is imported from the root.
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT.name}")

In [ ]:
import logging
import re
import time
from uuid import uuid4

from IPython.display import Markdown, display
from langchain_core.tracers.langchain import wait_for_all_tracers

from graph.llm import ModelGateway
from graph.nodes.researcher import RESULTS_PER_SUB_QUESTION, TavilySearchClient
from graph.prompts import planner as planner_prompt
from graph.prompts import writer as writer_prompts
from graph.report import cited_finding_ids, render_brief
from graph.supervisor import build_graph, research_run_config
from settings import settings


class RunLogRecorder(logging.Handler):
    """Keeps the graph's log messages, so retries and removed markers can be counted."""

    def __init__(self) -> None:
        super().__init__(level=logging.INFO)
        self.messages: list[str] = []

    def emit(self, record: logging.LogRecord) -> None:
        self.messages.append(record.getMessage())


# Every module under graph/ logs to a child of the "graph" logger.
graph_logger = logging.getLogger("graph")
graph_logger.setLevel(logging.INFO)
for old_recorder in [h for h in graph_logger.handlers if isinstance(h, RunLogRecorder)]:
    graph_logger.removeHandler(old_recorder)  # re-running this cell must not count twice
run_log = RunLogRecorder()
graph_logger.addHandler(run_log)

In [ ]:
TOPIC = "Due diligence on Symbotic: business model, customers, and main risks"

graph = build_graph(ModelGateway(), TavilySearchClient())
# CONCEPT: tracing-from-day-one. The run id is chosen before the run, so the
# trace can be looked up afterwards (cell below).
run_id = uuid4()
run_log.messages.clear()
started_at = time.perf_counter()

final_state = await graph.ainvoke(
    {"topic": TOPIC, "document_ids": []},
    config=research_run_config(TOPIC, run_id),
)

elapsed_seconds = time.perf_counter() - started_at
print(f"Done in {elapsed_seconds:.1f} s")

In [ ]:
brief_markdown = render_brief(TOPIC, final_state["draft"], final_state["findings"])
display(Markdown(brief_markdown))

In [ ]:
def count_messages(fragment: str) -> int:
    return sum(fragment in message for message in run_log.messages)


def count_multi_id_markers() -> int:
    counts = re.findall(r"markers citing more than one finding: (\d+)", " ".join(run_log.messages))
    return sum(int(count) for count in counts)


findings = final_state["findings"]
fetched_ids = {finding.id for finding in findings if finding.found}
run_numbers = {
    "sub-questions": len(final_state["sub_questions"]),
    "findings found": len(fetched_ids),
    "no evidence": sum(not finding.found for finding in findings),
    "sources cited": len(cited_finding_ids(final_state["draft"]) & fetched_ids),
    "sections": len(final_state["draft"]),
    "sections added for missed sub-questions": count_messages("outline missed"),
    "unknown markers removed": count_messages("removed unknown markers"),
    "multi-id markers": count_multi_id_markers(),
    "schema retries": count_messages("did not fit"),
    "transient retries": count_messages("transient error"),
    "seconds": round(elapsed_seconds, 1),
}
for measure, value in run_numbers.items():
    print(f"{measure:<42}{value}")

In [ ]:
# CONCEPT: tracing-from-day-one. LangSmith uploads runs from a background
# thread; wait until everything is sent, then find this run by its id.
wait_for_all_tracers()

if settings.langsmith_tracing:
    from langsmith import Client

    langsmith_client = Client()
    for lookup_attempt in range(1, 6):
        try:
            traced_run = langsmith_client.read_run(run_id)
            trace_url = langsmith_client.get_run_url(
                run=traced_run, project_name=settings.langsmith_project
            )
            print(f"Trace: {trace_url}")
            break
        except Exception:  # a new run can take a few seconds to appear
            time.sleep(2 * lookup_attempt)
    else:
        print(f"Trace not found yet: search project {settings.langsmith_project} for {run_id}")
else:
    print("Tracing is off (LANGSMITH_TRACING=false)")

In [ ]:
# The committed record of the run. Only the brief and its provenance are
# written: never settings, keys, or LangSmith ids (option C, three layers).
prompt_versions = (
    f"P-01 {planner_prompt.VERSION}, P-03 {writer_prompts.OUTLINE_VERSION}, "
    f"P-04 {writer_prompts.SECTION_VERSION}"
)
provenance_lines = [
    "> - Generated by `notebooks/phase_01.ipynb`, Phase 1.",
    f"> - Prompts: {prompt_versions}.",
    f"> - Models: `{settings.vre_chat_model}` (planner), `{settings.vre_writer_model}` (writer).",
    f"> - Search: Tavily, {RESULTS_PER_SUB_QUESTION} results per sub-question.",
    f"> - Run: {run_numbers['sub-questions']} sub-questions, "
    f"{run_numbers['findings found']} findings, {run_numbers['sections']} sections, "
    f"{run_numbers['seconds']} s.",
    "> - Phase 1: each citation names the passage its sentence was written from, but citations "
    "are not yet verified. Claim checking arrives in phase 5.",
]
title_line, brief_body = brief_markdown.split("\n", 1)
brief_file_text = "\n".join([title_line, "", *provenance_lines, brief_body])

brief_path = PROJECT_ROOT / "notebooks" / "phase_01_brief.md"
brief_path.write_text(brief_file_text, encoding="utf-8")
print(f"Wrote {brief_path.relative_to(PROJECT_ROOT)} ({len(brief_file_text.splitlines())} lines)")